# 03. Ray Tune with Ray Train
© 2026, Anyscale. All Rights Reserved

Notebook 02 trained one model with hyperparameters you picked by hand. This notebook sweeps those hyperparameters with Ray Tune, where every trial is itself the distributed training run from notebook 02. It teaches exactly `src/tune_ray_train.py`.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook </b>

<ol>
  <li>Why tune, and when</li>
  <li>Ray Tune on a toy problem</li>
  <li>Ray Tune concepts</li>
  <li>The shape: Tune driving Train</li>
  <li>The search space</li>
  <li>Resource math</li>
  <li>Run the search</li>
  <li>Results and the best checkpoint</li>
  <li>Promote the winner</li>
  <li>Fault tolerance at two levels</li>
  <li>Activity: sweep the epoch count too</li>
  <li>Ray Tune in production</li>
</ol>

</div>

**Setup**

In [ ]:
import dataclasses
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.fs
import torch

import ray
import ray.train
import ray.tune
from ray.train import Checkpoint, CheckpointConfig, FailureConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from ray.tune.integration.ray_train import TuneReportCallback
from ray.tune.schedulers import ASHAScheduler

sys.path.insert(0, os.path.abspath(".."))
from src.data import MNIST_TRANSFORM, raw_mnist
from src.model import build_resnet18
from src.settings import Settings, ray_init_with_repo

In [ ]:
settings = Settings.from_env()
ray_init_with_repo()
settings.describe()

## 1. Why tune, and when

|Challenge|Detail|Solution|
|---|---|---|
|**Searching many hyperparameters efficiently**|Trying learning rates and batch sizes one at a time, sequentially, wastes wall-clock time|Ray Tune schedules and runs many trials in parallel across the whole Ray cluster|
|**Wasting compute on trials that are clearly bad**|Running every trial to completion spends full training time even on hyperparameters that are failing early|Schedulers like ASHA watch intermediate metrics and stop underperforming trials early, freeing resources for promising ones|
|**Tracking many trials and their results**|Comparing configs, metrics, and checkpoints across dozens of trials by hand does not scale|Ray Tune tracks every trial's config, metrics, and checkpoints centrally, queryable with `get_best_result()` and `get_dataframe()`|
|**A trial that is not a single process**|A hyperparameter sweep over a distributed training job cannot just fork one process per trial|Ray Tune's driver-function pattern makes each trial itself a distributed job|

## 2. Ray Tune on a toy problem

Before sweeping the real training loop, here is Ray Tune on the smallest possible example: fitting a single scalar `a` in `predictions = distance * a`.

In [ ]:
def train_toy_model(config: dict) -> None:
    distances = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
    total_amounts = distances * 10
    predictions = distances * config["a"]
    rmse = np.sqrt(np.mean((total_amounts - predictions) ** 2))
    ray.tune.report({"rmse": rmse})

<div class="alert alert-block alert-warning">

The training function must accept a single <code>config</code> argument. Ray Tune passes each trial's hyperparameters in as this dictionary.

</div>

In [ ]:
toy_tuner = ray.tune.Tuner(
    train_toy_model,
    param_space={"a": ray.tune.randint(0, 20)},
    tune_config=ray.tune.TuneConfig(metric="rmse", mode="min", num_samples=5),
)
toy_results = toy_tuner.fit()

In [ ]:
toy_results.get_best_result().config

A `Tuner` takes a trainable function (`train_toy_model`), a search space (`param_space`), a metric and direction to optimize (`metric`, `mode`), and `num_samples`, the number of trials to run. `tuner.fit()` runs the trials and returns a `ResultGrid`.

## 3. Ray Tune concepts

Three defaults were implicit above. Made explicit, they look like this:

* **Resources per trial**: by default, one trial gets 1 CPU core. `ray.tune.with_resources` makes this explicit, and is how you would ask for a GPU per trial instead.
* **Search algorithm**: by default, `BasicVariantGenerator`, which is random or grid search over `param_space`.
* **Scheduler**: by default, `FIFOScheduler`, which runs every trial to completion in submission order with no early stopping.

In [ ]:
explicit_tuner = ray.tune.Tuner(
    ray.tune.with_resources(train_toy_model, {"cpu": 1}),
    param_space={"a": ray.tune.randint(0, 20)},
    tune_config=ray.tune.TuneConfig(
        mode="min",
        metric="rmse",
        num_samples=5,
        search_alg=ray.tune.search.BasicVariantGenerator(),
        scheduler=ray.tune.schedulers.FIFOScheduler(),
    ),
)
explicit_results = explicit_tuner.fit()

|<img src="https://docs.ray.io/en/latest/_images/tune_flow.png" width="800">|
|:--|
|How a Tuner, its search algorithm, its scheduler, and its trials fit together.|

## 4. The shape: Tune driving Train

Notebook 02's training loop, with the resume and deliberate-failure teaching aids stripped out for clarity, looked like this:

```python
def train_loop_per_worker(config: dict) -> None:
    ctx = ray.train.get_context()
    model = ray.train.torch.prepare_model(build_resnet18())
    optimizer = Adam(model.parameters(), lr=config["lr"])
    data_loader = ray.train.torch.prepare_data_loader(
        build_data_loader(config["data_root"], config["global_batch_size"] // ctx.get_world_size(), config["subset_size"])
    )
    for epoch in range(config["num_epochs"]):
        # ... forward, loss, backward, step ...
        report_checkpoint(model, optimizer, metrics, epoch)
```

The real function, unchanged, still has the resume and failure logic in it. Import it directly rather than redefining it, so this notebook trains with exactly the same code notebook 02 did:

In [ ]:
from src.train_ray_train import build_trainer, default_run_name, train_loop_per_worker

Ray Train v2 has no `Tuner(trainer)`: you cannot hand a `Trainer` object straight to a `Tuner`. Instead, a small **driver function** builds a `TorchTrainer` and calls `.fit()` itself, once per trial. `TuneReportCallback`, attached to the trainer's `RunConfig`, forwards the worker's metrics, and the checkpoint's path, up to Tune.

In [ ]:
def train_driver_fn(config: dict) -> None:
    """Runs once per Tune trial, on 1 CPU. Launches the distributed Train run and waits for it."""
    trial_id = ray.tune.get_context().get_trial_id()
    trainer = TorchTrainer(
        train_loop_per_worker,
        train_loop_config=config["train_loop_config"],
        scaling_config=ScalingConfig(num_workers=config["num_workers"], use_gpu=config["use_gpu"]),
        run_config=RunConfig(
            name=f"train-trial_id={trial_id}",  # stable name, so a restarted driver resumes in place
            storage_path=config["storage_path"],
            callbacks=[TuneReportCallback()],  # metrics + checkpoint path flow up to Tune
            checkpoint_config=CheckpointConfig(
                num_to_keep=1, checkpoint_score_attribute="loss", checkpoint_score_order="min"
            ),
            failure_config=FailureConfig(max_failures=2),  # worker-level retries
        ),
    )
    trainer.fit()

<div class="alert alert-block alert-warning">

<code>TuneReportCallback</code> only works inside a Tune trial. Calling <code>train_driver_fn</code> directly, outside a <code>Tuner</code>, raises an error because there is no trial context for it to report into.

</div>

|<img src="https://docs.ray.io/en/latest/_images/train_tune_interop.png" width="800">|
|:--|
|How a Tune trial (running `train_driver_fn`) drives a nested Ray Train run, and how metrics and checkpoints flow back up.|

`run_config.name=f"train-trial_id={trial_id}"` gives every trial's underlying Train run a stable, unique name: stable so a restarted driver resumes the same Train run in place, unique so trials never collide on the same storage path.

## 5. The search space

`build_param_space` mixes fixed values with sampled ones in a single `param_space`, nesting everything the training function needs under `train_loop_config`.

In [ ]:
def build_param_space(settings: Settings) -> dict:
    train_loop_config = settings.as_train_loop_config()
    train_loop_config["lr"] = ray.tune.loguniform(1e-4, 1e-2)  # the only sampled parts
    train_loop_config["global_batch_size"] = ray.tune.choice([64, 128, 256])
    return {
        "num_workers": settings.num_workers,
        "use_gpu": settings.use_gpu,
        "storage_path": settings.storage_path,
        "train_loop_config": train_loop_config,
    }

`ray.tune.loguniform` samples learning rates evenly across orders of magnitude, appropriate since a good learning rate is rarely known to better than an order of magnitude in advance. `ray.tune.choice` samples from a fixed set of options.

## 6. Resource math

Every trial's driver launches its own `TorchTrainer` with `num_workers` GPU workers. Running `max_concurrent_trials` of those at once means peak GPU usage is their product, not just `num_workers`.

In [ ]:
GPU_BUDGET = 4
num_workers = settings.num_workers
max_concurrent_trials = GPU_BUDGET // num_workers
peak_gpus = max_concurrent_trials * num_workers
print(f"max_concurrent_trials={max_concurrent_trials} num_workers={num_workers} -> peak {peak_gpus} GPUs")

<div class="alert alert-block alert-info">

Each trial also runs a 1-CPU driver process (<code>train_driver_fn</code> itself), separate from its GPU workers. Those drivers land on the head node, and that is exactly where you want them: the head is not preemptible, so a driver sitting there survives GPU worker pre-emption and handles the retries in section 10, instead of disappearing along with the workers it is supervising.

</div>

## 7. Run the search

In [ ]:
def build_tuner(settings: Settings, experiment_name: str, num_samples: int, max_concurrent_trials: int) -> ray.tune.Tuner:
    return ray.tune.Tuner(
        train_driver_fn,
        param_space=build_param_space(settings),
        tune_config=ray.tune.TuneConfig(
            metric="loss",
            mode="min",
            num_samples=num_samples,
            max_concurrent_trials=max_concurrent_trials,  # peak GPUs = this x num_workers
            scheduler=ASHAScheduler(
                time_attr="training_iteration",  # one Train report = one iteration
                grace_period=1,
                max_t=settings.num_epochs,
                reduction_factor=2,
            ),
        ),
        run_config=ray.tune.RunConfig(
            name=experiment_name,
            storage_path=settings.storage_path,
            failure_config=ray.tune.FailureConfig(max_failures=1),  # driver-level retries
        ),
    )

In [ ]:
import datetime

name = "tune-mnist-" + datetime.datetime.now(datetime.UTC).strftime("%Y%m%d-%H%M%S")
num_samples = 4
print(f"experiment={name} trials={num_samples} concurrent={max_concurrent_trials} peak_workers={peak_gpus}")
results = build_tuner(settings, name, num_samples, max_concurrent_trials).fit()

`ASHAScheduler` stops trials early based on intermediate performance: `grace_period=1` guarantees every trial gets at least one reported iteration before it can be stopped, and `reduction_factor=2` halves the surviving trial pool at each rung.

## 8. Results and the best checkpoint

In [ ]:
df = results.get_dataframe()
cols = [
    c
    for c in [
        "trial_id",
        "loss",
        "epoch",
        "training_iteration",
        "config/train_loop_config/lr",
        "config/train_loop_config/global_batch_size",
    ]
    if c in df.columns
]
df[cols].sort_values("loss")

Trials with a lower `training_iteration` than `settings.num_epochs` were stopped early by ASHA, not crashed: their last reported loss simply ranked poorly against other trials at the same rung.

In [ ]:
best = results.get_best_result()
print("best hyperparameters:", {k: best.config["train_loop_config"][k] for k in ("lr", "global_batch_size")})

In [ ]:
def best_checkpoint(results: ray.tune.ResultGrid, storage_path: str) -> Checkpoint:
    """TuneReportCallback attaches the Train checkpoint path as the metric `checkpoint_path`.

    That path is scheme-less on cloud storage, because Ray strips the URI scheme
    (e.g. "s3://") before handing it to Tune: the scheme belongs on Checkpoint.filesystem,
    not baked into the path string. `Checkpoint(path=...)` with no filesystem asks pyarrow
    to infer one from the path itself, and on a scheme-less cloud path that fails with
    "URI has empty scheme". It never shows up on local storage, where paths need no scheme
    either way, which is why a local smoke test can't catch it. The fix is to hand Checkpoint
    the filesystem explicitly, derived from the run's own storage_path (which DOES still
    have its scheme), instead of trying to recover it from the path.
    """
    best = results.get_best_result()
    path = best.metrics["checkpoint_path"]
    filesystem, _ = pyarrow.fs.FileSystem.from_uri(storage_path)
    return Checkpoint(path=path, filesystem=filesystem)

<div class="alert alert-block alert-warning">

<b>This bites on cloud storage, never locally.</b> <code>best.metrics["checkpoint_path"]</code> is a plain filesystem-looking path with no <code>s3://</code> or <code>gs://</code> prefix. Passing it straight to <code>Checkpoint(path=path)</code> works while <code>storage_path</code> is a local directory, then raises <code>pyarrow.lib.ArrowInvalid: URI has empty scheme</code> the moment <code>storage_path</code> points at cloud storage instead, because there is no scheme left in the path for pyarrow to infer a filesystem from. Reconstructing the filesystem from <code>storage_path</code> with <code>pyarrow.fs.FileSystem.from_uri</code>, and passing it alongside the path, fixes it in both places.

</div>

In [ ]:
checkpoint = best_checkpoint(results, settings.storage_path)
print("best checkpoint:", checkpoint)

Load it and generate predictions on nine images, the same way notebook 02 did.

In [ ]:
with checkpoint.as_directory() as ckpt_dir:
    state_dict = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu")

# Constructing a Checkpoint proves nothing by itself: prove it actually reads back.
num_params = sum(p.numel() for p in state_dict.values())
print(f"checkpoint loads: {len(state_dict)} tensors, {num_params:,} parameters")

loaded_model = build_resnet18()
loaded_model.load_state_dict(state_dict)
loaded_model.eval()

dataset = raw_mnist(settings.data_root, train=True)
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3

for i in range(1, cols * rows + 1):
    sample_idx = np.random.randint(0, len(dataset.data))
    img, label = dataset[sample_idx]
    normalized_img = MNIST_TRANSFORM(img)

    with torch.no_grad():
        prediction = loaded_model(normalized_img.unsqueeze(0)).argmax().item()

    figure.add_subplot(rows, cols, i)
    plt.title(f"label: {label}; pred: {prediction}")
    plt.axis("off")
    plt.imshow(img, cmap="gray")

## 9. Promote the winner

The sweep's best hyperparameters are just a `lr` and a `global_batch_size`. Promoting the winner means one full, un-pruned run with those values and the same training function this whole notebook has been using.

In [ ]:
best_lr = best.config["train_loop_config"]["lr"]
best_batch_size = best.config["train_loop_config"]["global_batch_size"]
promoted_settings = dataclasses.replace(settings, lr=best_lr, global_batch_size=best_batch_size)
promoted_result = build_trainer(promoted_settings, "promoted-" + default_run_name()).fit()
print("promoted run final metrics:", promoted_result.metrics)

## 10. Fault tolerance at two levels

This sweep is fault-tolerant twice over, at two different layers:

1. **Worker-level, inside a trial.** `train_driver_fn`'s `RunConfig(failure_config=FailureConfig(max_failures=2))` is the same automatic-retry mechanism from notebook 02: if a GPU worker crashes mid-trial, Ray Train restarts that trial's worker group and resumes from its last checkpoint.
2. **Driver-level, across trials.** `build_tuner`'s `ray.tune.RunConfig(failure_config=ray.tune.FailureConfig(max_failures=1))` covers the driver process itself: if the 1-CPU actor running `train_driver_fn` dies, Tune restarts it.

If the whole sweep is interrupted, for example the workspace restarts, resume it explicitly with `Tuner.restore`:

In [ ]:
restored_tuner = ray.tune.Tuner.restore(
    path=os.path.join(settings.storage_path, name),
    trainable=train_driver_fn,
    resume_errored=True,
)
restored_results = restored_tuner.fit()

<div class="alert alert-block alert-warning">

A plain <code>Tuner()</code> never resumes, no matter what you name the experiment. Setting <code>EXPERIMENT_NAME</code> to a value you used before does <b>not</b> resume that sweep; it just points a brand-new <code>Tuner</code> at the same storage path, where it will error on the existing directory or start overwriting it, depending on version. Only an explicit <code>Tuner.restore(path, trainable, ...)</code> call resumes a sweep. This is different from Ray Train in notebook 02, where reusing a `RunConfig(name, storage_path)` genuinely does resume, with no separate restore call. Also note Ray Train's own <code>Trainer.restore()</code> from v1 is gone entirely in v2; see <code>docs/ray-train-v1-to-v2.md</code>.

</div>

## 11. Activity: sweep the epoch count too

<div class="alert alert-block alert-info">

1. Add `num_epochs` to the search space, over `ray.tune.choice([1, 2])`.
2. Run 3 trials, one at a time (`max_concurrent_trials=1`).
3. Look at the results table: what happened to the trials that sampled `num_epochs=1`, compared to the ones that sampled `num_epochs=2`?

Use this snippet to guide you:

```python
activity_param_space = build_param_space(settings)
activity_param_space["train_loop_config"]["num_epochs"] = ...

activity_tuner = ray.tune.Tuner(
    train_driver_fn,
    param_space=activity_param_space,
    tune_config=ray.tune.TuneConfig(
        metric="loss", mode="min", num_samples=..., max_concurrent_trials=...,
        scheduler=ASHAScheduler(time_attr="training_iteration", grace_period=1, max_t=2, reduction_factor=2),
    ),
    run_config=ray.tune.RunConfig(name=..., storage_path=settings.storage_path),
)
activity_results = activity_tuner.fit()
activity_results.get_dataframe()[["trial_id", "loss", "training_iteration"]]
```

</div>

In [ ]:
# Write your solution here


<div class="alert alert-block alert-info">

<details>

<summary> Click here to see the solution </summary>

```python
activity_param_space = build_param_space(settings)
activity_param_space["train_loop_config"]["num_epochs"] = ray.tune.choice([1, 2])

activity_name = "tune-mnist-epochs-" + datetime.datetime.now(datetime.UTC).strftime("%Y%m%d-%H%M%S")
activity_tuner = ray.tune.Tuner(
    train_driver_fn,
    param_space=activity_param_space,
    tune_config=ray.tune.TuneConfig(
        metric="loss",
        mode="min",
        num_samples=3,
        max_concurrent_trials=1,
        scheduler=ASHAScheduler(time_attr="training_iteration", grace_period=1, max_t=2, reduction_factor=2),
    ),
    run_config=ray.tune.RunConfig(name=activity_name, storage_path=settings.storage_path),
)
activity_results = activity_tuner.fit()
activity_results.get_dataframe()[["trial_id", "loss", "training_iteration"]]
```

A trial that sampled `num_epochs=1` reports exactly one iteration and then finishes on its own; ASHA never gets a chance to stop it early, because it never had a second iteration to be compared at. A trial that sampled `num_epochs=2` is the one ASHA can actually prune: if its iteration-1 loss ranks in the worse half of the rung, ASHA stops it before it reaches iteration 2. Running one trial at a time also makes the order of these outcomes easy to read straight off the console, without interleaved output from concurrent trials.

</details>

</div>

## 12. Ray Tune in production

1. Uber's internal autotune service uses Ray Tune at scale. Read the [Anyscale blog post](https://www.anyscale.com/blog/how-uber-uses-ray-to-optimize-model-training).
2. Spotify uses Ray Tune for hyperparameter tuning across its ML platform. Read the [Spotify engineering post](https://engineering.atspotify.com/2023/02/unleashing-ml-innovation-at-spotify-with-ray/).

Next: notebook 04 takes this same code out of the notebook and runs it as an Anyscale Job, from the workspace and from a laptop.

## Further reading

| Topic | Link |
|---|---|
| Tune key concepts | https://docs.ray.io/en/latest/tune/key-concepts.html |
| Search space tutorial | https://docs.ray.io/en/latest/tune/tutorials/tune-search-spaces.html |
| Schedulers | https://docs.ray.io/en/latest/tune/api/schedulers.html |
| Stopping | https://docs.ray.io/en/latest/tune/tutorials/tune-stopping.html |
| Train hyperparameter optimization guide | https://docs.ray.io/en/latest/train/user-guides/hyperparameter-optimization.html |
| Template: ray-tune-train-integration | https://github.com/anyscale/templates/tree/main/templates/ray-tune-train-integration |
| Template: tune_pytorch_asha | https://github.com/anyscale/templates/tree/main/templates/tune_pytorch_asha |
| Template: parallel-experiments | https://github.com/anyscale/templates/tree/main/templates/parallel-experiments |